# Lab 2 · Exercise C — Pulls, bias and coverage

Test your fit with **toy experiments**: generate with known truth, fit, and calculate the **pull** of the signal yield:

$$ \text{pull} = \frac{\hat{\nu}_s - \nu_s^{\rm true}}{\sigma_{\hat{\nu}_s}} $$

Here $\nu_s$ is `nsig`. Instead of writing the loop by hand, we use **`RooMCStudy`**: it repeats *generate → fit* for you, resets the parameters at every toy, and keeps the results.

In [ ]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.gErrorIgnoreLevel = ROOT.kWarning                          # no Info messages from Minuit2
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)
ROOT.RooMsgService.instance().getStream(0).minLevel = ROOT.RooFit.WARNING   # no progress lines from RooMCStudy

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
m     = ROOT.RooRealVar("m", "m [MeV]", 5080, 5540)
mean  = ROOT.RooRealVar("mean", "mean", 5288, 5200, 5350)
sigma = ROOT.RooRealVar("sigma", "sigma", 14.8, 5, 50)
slope = ROOT.RooRealVar("slope", "slope", -0.002, -0.02, 0)
sig   = ROOT.RooGaussian("sig", "sig", m, mean, sigma)
bkg   = ROOT.RooExponential("bkg", "bkg", m, slope)
nsig  = ROOT.RooRealVar("nsig", "signal yield", 33, 0, 400)
nbkg  = ROOT.RooRealVar("nbkg", "background yield", 133, 0, 600)
model = ROOT.RooAddPdf("model", "sig+bkg", ROOT.RooArgList(sig, bkg), ROOT.RooArgList(nsig, nbkg))

TRUTH = {mean: 5288, sigma: 14.8, slope: -0.002, nbkg: 133}

def set_truth(ns_true):
    """Put every parameter at the truth. RooMCStudy generates with the current values of the parameters."""
    for v, t in TRUTH.items():
        v.setVal(t)
    nsig.setVal(ns_true)

## C1 · The toy study
Create a `RooMCStudy` for the model, with the observable `m`, in **extended** mode (the number of events fluctuates, as in the data), and run it on `NTOYS` toys.
`RooFit.Silence()` keeps the output short. The study **generates** with the current values of the parameters, and **fits** each toy starting from them.

In [ ]:
NTOYS = 1000
set_truth(33)          # generate with nsig = 33 and the true values of the other parameters

mcs = None             # TODO: ROOT.RooMCStudy(model, ROOT.RooArgSet(m), ROOT.RooFit.Silence(), ROOT.RooFit.Extended(True))
# TODO: mcs.generateAndFit(NTOYS)

In [ ]:
def pull_stats(mcs, name, ntoys):
    """Pull statistics of the parameter `name`, from the dataset of the fit parameters of a RooMCStudy."""
    ds = mcs.fitParDataSet()            # one row per toy whose fit converged
    n = ds.numEntries()
    pull = np.array([ds.get(i).getRealValue(name + "pull") for i in range(n)])   # (fitted - generated) / error
    val  = np.array([ds.get(i).getRealValue(name)          for i in range(n)])
    return dict(n=n, failed=ntoys - n, mean=pull.mean(), rms=pull.std(),
                coverage=np.mean(np.abs(pull) < 1), at_boundary=np.mean(val < 0.01))

def report(label, s):
    print(f"{label}: fits that converged = {s['n']}, failed fits = {s['failed']}")
    print(f"   pull mean = {s['mean']:+.3f}   pull RMS = {s['rms']:.3f}   coverage |pull| < 1 = {s['coverage']:.3f}")
    print(f"   fraction of toys with nsig < 0.01 = {s['at_boundary']:.3f}")

frame = mcs.plotPull(nsig, ROOT.RooFit.FitGauss(True))      # histogram of the pulls, with a Gaussian fit
c1 = ROOT.TCanvas("c1", "pulls, nsig = 33", 700, 500)
frame.Draw()
c1.Draw()

s33 = pull_stats(mcs, "nsig", NTOYS)
report("C1, nsig = 33", s33)
check("C1: the pull mean is compatible with 0", abs(s33["mean"]) < 0.2)
check("C1: the pull RMS is close to 1", 0.9 < s33["rms"] < 1.25)
check("C1: the coverage is close to 68%", 0.62 < s33["coverage"] < 0.75)

## C2 · Low statistics
Repeat the study with $\nu_s=5$: put the parameters at the truth with `set_truth(5)`, create a new `RooMCStudy` (`mcs5`) and run it.
Compare with C1: the **pull distribution** (mean, RMS, shape), the number of **failed fits**, the fraction of toys with `nsig` at the **lower boundary**, and the coverage.

In [ ]:
set_truth(5)

mcs5 = None            # TODO: a new RooMCStudy, as in C1
# TODO: mcs5.generateAndFit(NTOYS)

In [ ]:
frame5 = mcs5.plotPull(nsig, ROOT.RooFit.FitGauss(True))
c5 = ROOT.TCanvas("c5", "pulls, nsig = 5", 700, 500)
frame5.Draw()
c5.Draw()

s5 = pull_stats(mcs5, "nsig", NTOYS)
report("C2, nsig = 5", s5)
check("C2: the study was run with nsig = 5 (narrower pulls than in C1)", s5["rms"] < 0.95)

## Questions (optional)
Write your answer under each question, in 2–3 lines, and quote the numbers or plots of the steps indicated.
1. **(C1)** For $\nu_s=33$: is the fit biased (look at the mean of the pulls)? Are the HESSE errors reliable (look at the RMS of the pulls and the coverage)? Write the numbers.
2. **(C2)** For $\nu_s=5$: what is different in the pull distribution compared with C1 (mean, RMS, shape), and why? What would you report instead of a symmetric error?
3. **(C1, C2)** A fit that does not converge is not in the dataset of the study: how do you count the failed fits? Why is it important to count them, and what bias do you introduce if you ignore them?